# CS 340 Project Two Dashboard
Student: Shaban Ghaith


In [ ]:
# CS 340 Project Two Dashboard
# Student: Shaban Ghaith

from jupyter_dash import JupyterDash

import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
JupyterDash.infer_jupyter_proxy_config()

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use the CRUD file that already exists in Codio.
from CRUD_Python_Module import CRUD

username = "aacuser"
password = "YOUR_PASSWORD"


def clean_dataframe(records):
    df = pd.DataFrame.from_records(records)
    for col in ["_id", "Unnamed: 0"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)
    return df


def load_data():
    # Try the database names commonly used in CS 340.
    for database_name in ["AAC", "aac"]:
        try:
            db = CRUD(username, password, database=database_name, collection="animals")
            records = db.read({})
            if records:
                print(f"Loaded {len(records)} records from MongoDB database {database_name}")
                return clean_dataframe(records)
        except Exception as exc:
            print(f"Could not load MongoDB database {database_name}: {exc}")

    # Backup path for Codio if Mongo is not responding.
    if os.path.exists("aac_shelter_outcomes.csv"):
        df = pd.read_csv("aac_shelter_outcomes.csv")
        if "Unnamed: 0" in df.columns:
            df.drop(columns=["Unnamed: 0"], inplace=True)
        print(f"Loaded {len(df)} records from CSV fallback")
        return df

    print("No data found. Check MongoDB or upload aac_shelter_outcomes.csv into this folder.")
    return pd.DataFrame()


df = load_data()

if "age_upon_outcome_in_weeks" in df.columns:
    df["age_upon_outcome_in_weeks"] = pd.to_numeric(df["age_upon_outcome_in_weeks"], errors="coerce")
if "location_lat" in df.columns:
    df["location_lat"] = pd.to_numeric(df["location_lat"], errors="coerce")
if "location_long" in df.columns:
    df["location_long"] = pd.to_numeric(df["location_long"], errors="coerce")


def filter_dataframe(filter_type):
    dff = df.copy()
    if dff.empty:
        return dff

    if filter_type == "water":
        breeds = "Labrador Retriever|Chesapeake Bay Retriever|Newfoundland"
        return dff[
            (dff["animal_type"] == "Dog") &
            (dff["breed"].fillna("").str.contains(breeds, case=False, regex=True)) &
            (dff["sex_upon_outcome"] == "Intact Female") &
            (dff["age_upon_outcome_in_weeks"] >= 26) &
            (dff["age_upon_outcome_in_weeks"] <= 156)
        ]

    if filter_type == "mountain":
        breeds = "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler"
        return dff[
            (dff["animal_type"] == "Dog") &
            (dff["breed"].fillna("").str.contains(breeds, case=False, regex=True)) &
            (dff["sex_upon_outcome"] == "Intact Male") &
            (dff["age_upon_outcome_in_weeks"] >= 26) &
            (dff["age_upon_outcome_in_weeks"] <= 156)
        ]

    if filter_type == "disaster":
        breeds = "Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler"
        return dff[
            (dff["animal_type"] == "Dog") &
            (dff["breed"].fillna("").str.contains(breeds, case=False, regex=True)) &
            (dff["sex_upon_outcome"] == "Intact Male") &
            (dff["age_upon_outcome_in_weeks"] >= 20) &
            (dff["age_upon_outcome_in_weeks"] <= 300)
        ]

    return dff


def find_logo_file():
    for filename in os.listdir("."):
        if filename.lower().endswith(".png") and "grazioso" in filename.lower():
            return filename
    return "Grazioso Salvare Logo.png"


image_filename = find_logo_file()
encoded_image = base64.b64encode(open(image_filename, "rb").read()).decode()

app = JupyterDash(__name__)

app.layout = html.Div([
    html.Div([
        html.A(
            html.Img(src="data:image/png;base64,{}".format(encoded_image), style={"height": "95px"}),
            href="https://www.snhu.edu",
            target="_blank"
        ),
        html.H1("Grazioso Salvare Rescue Animal Dashboard"),
        html.H3("Created by Shaban Ghaith | CS 340 Project Two"),
    ], style={"textAlign": "center"}),
    html.Hr(),

    dcc.RadioItems(
        id="filter-type",
        options=[
            {"label": "Reset - All Animals", "value": "reset"},
            {"label": "Water Rescue", "value": "water"},
            {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
            {"label": "Disaster or Individual Tracking", "value": "disaster"},
        ],
        value="reset",
        inline=True
    ),
    html.Br(),
    html.Div(id="record-count", style={"fontWeight": "bold"}),
    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "minWidth": "120px", "maxWidth": "250px", "whiteSpace": "normal"},
        style_header={"backgroundColor": "#1f4e79", "color": "white", "fontWeight": "bold"}
    ),

    html.Br(),
    html.Hr(),
    html.Div(className="row", style={"display": "flex"}, children=[
        html.Div(id="graph-id", className="col s12 m6", style={"width": "50%"}),
        html.Div(id="map-id", className="col s12 m6", style={"width": "50%"})
    ])
])


@app.callback(
    [Output("datatable-id", "data"), Output("record-count", "children")],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    dff = filter_dataframe(filter_type)
    labels = {
        "reset": "Reset - All Animals",
        "water": "Water Rescue",
        "mountain": "Mountain or Wilderness Rescue",
        "disaster": "Disaster or Individual Tracking",
    }
    return dff.to_dict("records"), f"{labels.get(filter_type, 'Reset')}: {len(dff):,} matching records"


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):
    dff = pd.DataFrame(viewData if viewData is not None else [])
    if dff.empty or "breed" not in dff.columns:
        return [html.P("No breed data available for this filter.")]

    breed_counts = dff["breed"].value_counts().nlargest(10).reset_index()
    breed_counts.columns = ["breed", "count"]
    return [dcc.Graph(figure=px.pie(breed_counts, names="breed", values="count", title="Top Rescue Candidate Breeds"))]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    return [{"if": {"column_id": i}, "background_color": "#D2F3FF"} for i in selected_columns]


@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"), Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None:
        return [html.P("No map data available.")]

    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return [html.P("No map data available for this filter.")]

    row = index[0] if index else 0
    if row >= len(dff):
        row = 0

    return [
        dl.Map(style={"width": "100%", "height": "500px"}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[dff.iloc[row]["location_lat"], dff.iloc[row]["location_long"]], children=[
                dl.Tooltip(str(dff.iloc[row]["breed"])),
                dl.Popup([
                    html.H3("Animal Name"),
                    html.P(str(dff.iloc[row]["name"]))
                ])
            ])
        ])
    ]


# This matches the original notebook launch style, which works better in Codio.
# If port 8050 is busy, stop the old kernel or change this to app.run_server(port=8051).
app.run_server()
